# ENGRAMA V3 × TinyStories — Modelo autorregresivo de ~20M con GPT-2 🧠⚡

Entrenamiento **completo** de un modelo de lenguaje autorregresivo **sin atención** (cero $QK^T$, cero softmax temporal) con la librería [ENGRAMA V3](https://github.com/bueormnew/engrama) instalada desde GitHub.

| Aspecto | Valor |
|---|---|
| Arquitectura | ENGRAMA V3 (sinapsis factorizadas, offsets diádicos, caché jerárquico) |
| Parámetros | **~20.3M** (verificado en ejecución; el notebook aborta si construye menos) |
| Datos | **TinyStories V2-GPT4 completo** (train 2.08 GiB + valid, descarga verificada) |
| Tokenizer | **GPT-2 BPE** (vocabulario 50,257) |
| Contexto | **512 tokens** |
| Salida | checkpoint `engrama_v3_20m_gpt2/` + muestras de inferencia |

> 🔧 **FAST_MODE = False por defecto** → entrenamiento real (GPU T4 o mejor).
> **FAST_MODE = True** → smoke test de todo el pipeline en minutos, incluso en CPU.
> 🖥️🖥️ **Multi-GPU portable**: con 2+ GPUs (p. ej. Kaggle **T4 x2**) el entrenamiento usa `DataParallel`,
> pero la pérdida se calcula **por GPU** (solo se reúnen escalares) y la agregación del evocador se
> procesa **por trozos de vocabulario con checkpointing** → pico de memoria ~3-4 GiB/GPU en vez de ~20 GiB.

- Autor: **BUEORM** · Licencia: **AGPL-3.0**


## 1️⃣ Instalación (ENGRAMA V3 desde GitHub + transformers)


In [ ]:
# ENGRAMA se instala SIEMPRE desde GitHub (el nombre 'engrama' en PyPI es otro paquete)
!pip install -q git+https://github.com/bueormnew/engrama.git transformers numpy

import os
import math
import time
import random
import glob
import json

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F

import engrama
from engrama import (EngramaConfig, EngramaModel, Generator,
                     save_model, load_model, chunked_cross_entropy)

print('ENGRAMA', engrama.__version__, '| torch', torch.__version__)
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
NGPU = torch.cuda.device_count() if DEVICE == 'cuda' else 0
if DEVICE == 'cuda':
    torch.backends.cudnn.benchmark = True
    print('GPUs:', [torch.cuda.get_device_name(i) for i in range(NGPU)])
else:
    print('Dispositivo: CPU  (el modo FULL es solo viable en GPU; usa FAST_MODE=True)')


## 2️⃣ Configuración central

Todos los hiperparámetros en un solo lugar. En **FULL** (`FAST_MODE=False`) se construye el modelo
real de **~20.3M** con contexto 512 y se entrena sobre el **dataset completo**; en **FAST** se reducen
modelo, datos y pasos solo para validar el pipeline de punta a punta.


In [ ]:
# ------------------------- MODO -----------------------------------------
FAST_MODE = False   # False => entrenamiento REAL ~20.3M / TinyStories completo / 512
SEED = 1234

# ------------------------- DATOS -----------------------------------------
SEQ_LEN = 512                 # contexto del modelo real (tokens GPT-2)
MAX_TRAIN_SEQS = None         # None = TODO el split de entrenamiento (~370k secuencias).
                              #   Pon p. ej. 80_000 para una corrida mas corta.
MAX_VALID_SEQS = 1_000        # secuencias de validacion (solo se usa en FAST)
TRAIN_FILE = 'tinystories_train.txt'
VALID_FILE = 'tinystories_valid.txt'
# Tamano EXACTO de los ficheros en Hugging Face (verificacion anti-truncado):
TRAIN_BYTES = 2_227_753_162   # TinyStoriesV2-GPT4-train.txt (~2.08 GiB)
VALID_BYTES = 22_502_601      # TinyStoriesV2-GPT4-valid.txt (~21.5 MiB)

# ------------------------- MODELO (~20.3M en FULL) ------------------------
MODEL_KW = dict(
    d_model=256, d_gate=32, d_ff=1024,
    num_cells=8, num_encoder_layers=2,
    num_consolidation_layers=9,   # L >= ceil(log2 512) => cobertura binaria completa
    num_candidates=4, candidate_aggregation='logsumexp',
    synapse_rank=32, version='v3', global_anchor=False,
)

# ------------------------- ENTRENAMIENTO ----------------------------------
# batch = 16 POR GPU. Con la loss por trozos ya no hace falta recortar el batch
# para que quepa en T4 de 16 GB: el pico es ~3-4 GiB/GPU en el modo FULL.
BATCH_SIZE = 16 * max(1, NGPU)
EVAL_BATCH = 8
EPOCHS = 1
LR = 6e-4
WEIGHT_DECAY = 0.01
GRAD_CLIP = 1.0
WARMUP_STEPS = 200
LOG_EVERY = 50
EVAL_EVERY = 250
SAMPLE_EVERY = 500
EVAL_BATCHES = 25
RESUME = True   # si existe checkpoint en SAVE_DIR, continua desde el paso guardado

if FAST_MODE:
    SEQ_LEN = 128
    MAX_TRAIN_SEQS = 256
    MAX_VALID_SEQS = 64
    MODEL_KW.update(d_model=128, d_gate=16, d_ff=512, num_cells=4,
                    num_encoder_layers=2, num_consolidation_layers=7,
                    num_candidates=2, synapse_rank=16)
    BATCH_SIZE, EVAL_BATCH, EPOCHS = 8, 8, 2
    WARMUP_STEPS = 10
    LOG_EVERY, EVAL_EVERY, SAMPLE_EVERY, EVAL_BATCHES = 10, 16, 16, 4

SAVE_DIR = ('/kaggle/working/engrama_v3_20m_gpt2'
            if os.path.isdir('/kaggle/working') else './engrama_v3_20m_gpt2')
random.seed(SEED)
torch.manual_seed(SEED)
print('Modo = %s | SEQ_LEN=%d | train_seqs=%s | batch=%d | ckpt -> %s' % (
    'FAST' if FAST_MODE else 'FULL', SEQ_LEN,
    'TODO el split' if MAX_TRAIN_SEQS is None else '<= %s' % MAX_TRAIN_SEQS,
    BATCH_SIZE, SAVE_DIR))


## 3️⃣ Datos: TinyStories completo (descarga robusta y verificada)

Descarga `TinyStoriesV2-GPT4-train.txt` / `-valid.txt` de Hugging Face con:

- **reintentos con backoff** y **reanudación** (Range) si la conexión se corta (fichero de ~2 GiB),
- **verificación de tamaño exacto**: si el fichero queda truncado se elimina y se relanza;
  **no hay fallbacks silenciosos en modo FULL** (antes, un corte de red te dejaba entrenando
  con el split de validación o un corpus sintético sin que te dieras cuenta).
- si montaste el dataset `roneneldan/TinyStories` en Kaggle, se usa directamente.

La lectura del corpus es **streaming por cuentos**: nunca se cargan ~2 GB en RAM.


In [ ]:
def iter_stories(path):
    """Generador streaming: produce cuentos separados por linea en blanco."""
    buf = []
    with open(path, 'r', encoding='utf-8', errors='ignore') as f:
        for line in f:
            if line.strip():
                buf.append(line.rstrip('\n'))
            elif buf:
                yield ' '.join(buf).strip()
                buf = []
    if buf:
        yield ' '.join(buf).strip()

def download_verified(url, path, expected_bytes, retries=8):
    """Descarga con reintentos, reanudacion (Range) y verificacion de tamano EXACTO."""
    import urllib.request
    done = os.path.getsize(path) if os.path.exists(path) else 0
    if done == expected_bytes:
        print('  %s: ya completo (%.0f MB)' % (path, done / 2**20))
        return path
    if 0 < done > expected_bytes:
        print('  %s: tamano incoherente, se redescarga desde cero' % path)
        os.remove(path)
        done = 0
    for attempt in range(1, retries + 1):
        mode = 'ab' if done else 'wb'
        headers = {'Range': 'bytes=%d-' % done} if done else {}
        try:
            req = urllib.request.Request(url, headers=headers)
            with urllib.request.urlopen(req, timeout=120) as resp:
                total = done if resp.status == 206 else 0
                if resp.status == 200:      # el servidor ignoro el Range
                    mode = 'wb'
                with open(path, mode) as f:
                    while True:
                        chunk = resp.read(4 * 1024 * 1024)
                        if not chunk:
                            break
                        f.write(chunk)
                        total += len(chunk)
                        if total % (256 * 1024 * 1024) < 4 * 1024 * 1024:
                            print('    %s: %.0f MB ...' % (path, total / 2**20))
            done = os.path.getsize(path)
            if done == expected_bytes:
                print('  %s: OK (%.0f MB verificados)' % (path, done / 2**20))
                return path
            print('  %s: incompleto (%d != %d); reintento ...' % (path, done, expected_bytes))
            if attempt >= retries:
                if os.path.exists(path):
                    os.remove(path)
                raise RuntimeError(
                    'Descarga de TinyStories fallida: el fichero quedo truncado '
                    '(%d != %d bytes) tras %d intentos.\n'
                    'Consejos: (1) en Kaggle activa "Internet" en el panel derecho; '
                    '(2) si la red es inestable, vuelve a ejecutar esta celda '
                    '(la descarga se reanuda sola); (3) o monta el dataset '
                    'roneneldan/TinyStories como input y esta celda lo usara directo.'
                    % (done, expected_bytes, retries))
        except Exception as exc:
            done = os.path.getsize(path) if os.path.exists(path) else 0
            if attempt >= retries:
                if os.path.exists(path):
                    os.remove(path)
                raise RuntimeError(
                    'Descarga de TinyStories fallida tras %d intentos (%s).\n'
                    'Detalle: %s\n'
                    'Consejos: (1) en Kaggle activa "Internet" en el panel derecho; '
                    '(2) si la red es inestable, vuelve a ejecutar esta celda '
                    '(la descarga se reanuda sola); (3) o monta el dataset '
                    'roneneldan/TinyStories como input y esta celda lo usara directo.'
                    % (retries, type(exc).__name__, exc)) from exc
            print('  %s: %s; reintento %d/%d ...' % (path, type(exc).__name__, attempt, retries))
            time.sleep(min(30.0, 2 ** attempt))
    raise RuntimeError('Descarga de %s no verificada' % path)

def find_kaggle_file(basename, expected):
    for pat in ('/kaggle/input/*/' + basename, '/kaggle/input/**/' + basename):
        for hit in sorted(glob.glob(pat, recursive=True)):
            if os.path.getsize(hit) == expected:
                return hit
    return None

TRAIN_URL = ('https://huggingface.co/datasets/roneneldan/TinyStories/resolve/main/'
             'TinyStoriesV2-GPT4-train.txt')
VALID_URL = ('https://huggingface.co/datasets/roneneldan/TinyStories/resolve/main/'
             'TinyStoriesV2-GPT4-valid.txt')

FALLBACK_STORY = (
    'Once upon a time there was a little cat named Lily. Lily liked to play in the '
    'garden with her red ball. One day a dog named Tom came and they played together '
    'all day. At night Lily went home, ate her dinner and slept. The end.'
)

train_path = valid_path = None
if FAST_MODE:
    print('FAST_MODE: basta el split de validacion (o corpus sintetico offline).')
    try:
        valid_path = download_verified(VALID_URL, VALID_FILE, VALID_BYTES)
        train_path = valid_path
    except Exception as exc:
        print('Aviso FAST_MODE sin acceso a TinyStories:', type(exc).__name__)
        with open('tinystories_synth.txt', 'w', encoding='utf-8') as f:
            f.write('\n\n'.join(FALLBACK_STORY for _ in range(600)))
        train_path = valid_path = 'tinystories_synth.txt'
        print('-> Corpus SINTETICO local (solo valida el pipeline).')
else:
    print('FULL: se requiere el dataset TinyStories COMPLETO (train ~2.08 GiB).')
    train_path = find_kaggle_file('TinyStoriesV2-GPT4-train.txt', TRAIN_BYTES)
    if train_path is None:
        train_path = download_verified(TRAIN_URL, TRAIN_FILE, TRAIN_BYTES)
    else:
        print('  train: dataset montado en Kaggle ->', train_path)
    valid_path = find_kaggle_file('TinyStoriesV2-GPT4-valid.txt', VALID_BYTES)
    if valid_path is None:
        valid_path = download_verified(VALID_URL, VALID_FILE, VALID_BYTES)
    else:
        print('  valid: dataset montado en Kaggle ->', valid_path)

print('train:', train_path, '| valid:', valid_path)


## 4️⃣ Tokenizer GPT-2 (BPE, vocabulario 50,257)

Adaptador con la interfaz exacta que espera `engrama.Generator` (`encode/decode/SPECIAL_TOKENS/vocab_size`).
En modo FULL el tokenizer GPT-2 es **obligatorio**: si no puede cargarse, el notebook aborta con
instrucciones (antes caía silenciosamente a un tokenizador de caracteres y el modelo resultante no era el anunciado).


In [ ]:
class GPT2Adapter:
    """Interfaz ENGRAMA sobre el tokenizer GPT-2 de Hugging Face."""
    def __init__(self, hf_tok):
        self.tok = hf_tok
        eot = hf_tok.eos_token_id  # 50256 <|endoftext|>
        self.SPECIAL_TOKENS = {'<eos>': eot, '<bos>': eot, '<pad>': eot}
        self.vocab_size = len(hf_tok)

    def encode(self, text, add_bos=False, add_eos=False):
        ids = list(self.tok.encode(text, add_special_tokens=False))
        if add_bos:
            ids = [self.SPECIAL_TOKENS['<bos>']] + ids
        if add_eos:
            ids = ids + [self.SPECIAL_TOKENS['<eos>']]
        return ids

    def decode(self, ids, skip_special_tokens=True):
        return self.tok.decode(list(ids), skip_special_tokens=skip_special_tokens)

    def encode_batch(self, texts):
        outs = self.tok(list(texts), add_special_tokens=False)['input_ids']
        return [list(ids) + [self.SPECIAL_TOKENS['<eos>']] for ids in outs]

tokenizer = None
_hf = None
try:
    os.environ.setdefault('HF_HUB_ETAG_TIMEOUT', '30')
    from transformers import GPT2TokenizerFast
    _hf = GPT2TokenizerFast.from_pretrained('gpt2')
    # sanity check: sin internet, from_pretrained puede devolver un stub vacio
    if len(_hf) != 50257 or not _hf.encode('hello world', add_special_tokens=False):
        raise RuntimeError('GPT-2 incompleto (vocab=%d)' % len(_hf))
    tokenizer = GPT2Adapter(_hf)
    print('Tokenizer: GPT-2 BPE, vocab =', tokenizer.vocab_size)
except Exception as exc:
    _hf = None   # un stub a medias NO debe filtrarse al checkpoint
    if not FAST_MODE:
        raise RuntimeError(
            'El tokenizer GPT-2 es OBLIGATORIO en modo FULL y no pudo cargarse.\n'
            'Causa: %s: %s\n'
            'Consejos: habilita Internet (transformers descarga gpt2 la primera vez) '
            'o ejecuta este notebook con FAST_MODE = True.'
            % (type(exc).__name__, exc)) from exc
    from engrama import EngramaTokenizer
    print('FAST_MODE: GPT-2 no disponible (%s) -> fallback char-level' % type(exc).__name__)
    sample = '\n'.join(s for _, s in zip(range(200), iter_stories(train_path)))
    tokenizer = EngramaTokenizer().fit_on_text(sample)
    print('Tokenizer: char-level fallback, vocab =', tokenizer.vocab_size)

VOCAB_SIZE = tokenizer.vocab_size
EOS_ID = tokenizer.SPECIAL_TOKENS['<eos>']
print(repr(tokenizer.decode(tokenizer.encode('Once upon a time, there was a cat.')[:12])))


## 5️⃣ Tokenización en streaming → memmap (sin reventar la RAM)

Cada cuento se tokeniza y se separa con `<|endoftext|>`; los ids se escriben **directamente a un
`np.memmap` int32 en disco** por lotes de 256 cuentos. Antes se acumulaban ~190M ids en listas de
Python (~5 GB de RAM) y ahí estaba uno de los errores de memoria; ahora el pico de RAM es el de un lote.


In [ ]:
class MemmapTokenWriter:
    """Escribe ids int32 en un memmap creciente (pico de RAM = 1 lote de cuentos)."""
    def __init__(self, out_raw, initial_capacity):
        self.path = out_raw
        self.cap = max(1024, int(initial_capacity))
        self.mm = np.memmap(out_raw, dtype=np.int32, mode='w+', shape=(self.cap,))
        self.pos = 0

    def ensure(self, extra):
        if self.pos + extra <= self.cap:
            return
        new_cap = max(self.cap * 2, self.pos + extra)
        self.mm.flush()
        del self.mm
        with open(self.path, 'r+b') as f:
            f.truncate(new_cap * 4)
        self.mm = np.memmap(self.path, dtype=np.int32, mode='r+', shape=(new_cap,))
        self.cap = new_cap

    def extend(self, ids):
        self.ensure(len(ids))
        self.mm[self.pos:self.pos + len(ids)] = np.asarray(ids, dtype=np.int32)
        self.pos += len(ids)

    def finalize(self):
        self.mm.flush()
        n = int(self.pos)
        del self.mm
        with open(self.path, 'r+b') as f:
            f.truncate(n * 4)
        return np.memmap(self.path, dtype=np.int32, mode='r', shape=(n,))

def _flush_batch(writer, tokenizer, batch):
    if hasattr(tokenizer, 'encode_batch'):
        batch_ids = tokenizer.encode_batch(batch)
    else:
        batch_ids = [tokenizer.encode(t, add_eos=True) for t in batch]
    for ids in batch_ids:
        if ids:
            writer.extend(ids)

def stories_to_memmap(path, out_raw, tokenizer, batch_stories=256, max_ids=None):
    """Tokeniza el corpus por lotes y lo deja en un memmap int32 en disco."""
    # GPT-2 BPE sobre ingles ~= 12 bytes/texto por token: 0.10 es margen holgado
    # (el writer crece solo si hiciera falta).
    writer = MemmapTokenWriter(out_raw, initial_capacity=os.path.getsize(path) * 0.10)
    batch, n_stories = [], 0
    for story in iter_stories(path):
        batch.append(story)
        if len(batch) < batch_stories:
            continue
        _flush_batch(writer, tokenizer, batch)
        n_stories += len(batch)
        batch = []
        if n_stories % 25000 < batch_stories:
            print('  %7d cuentos | %10d tokens ...' % (n_stories, writer.pos))
        if max_ids and writer.pos >= max_ids:
            break
    if batch and not (max_ids and writer.pos >= max_ids):
        _flush_batch(writer, tokenizer, batch)
        n_stories += len(batch)
    print('  %7d cuentos | %10d tokens' % (n_stories, writer.pos))
    return writer.finalize()

t0 = time.time()
max_train_ids = MAX_TRAIN_SEQS * (SEQ_LEN + 1) if MAX_TRAIN_SEQS else None
max_valid_ids = MAX_VALID_SEQS * (SEQ_LEN + 1) if MAX_VALID_SEQS else None
print('Tokenizando split de entrenamiento (streaming -> memmap) ...')
train_mm = stories_to_memmap(train_path, 'tinystories_train.ids', tokenizer,
                             max_ids=max_train_ids)
print('Tokenizando split de validacion ...')
valid_mm = stories_to_memmap(valid_path, 'tinystories_valid.ids', tokenizer,
                             max_ids=max_valid_ids)
print('Tokenizacion: %ds | train %s tokens | valid %s tokens'
      % (int(time.time() - t0), format(train_mm.shape[0], ','), format(valid_mm.shape[0], ',')))
assert train_mm.shape[0] > SEQ_LEN + 1, 'datos de entrenamiento insuficientes'
assert valid_mm.shape[0] > SEQ_LEN + 1, 'datos de validacion insuficientes'


## 6️⃣ Ventanas de 512 tokens y DataLoaders

El stream plano de ids se corta en ventanas contiguas de `SEQ_LEN + 1` (entrada + objetivo
desplazado, LM autorregresivo). El dataset son **vistas sobre el memmap**: cero copias en RAM.


In [ ]:
class TokenWindows(torch.utils.data.Dataset):
    """Ventanas contiguas de SEQ_LEN+1 sobre el memmap (vistas, sin copias)."""
    def __init__(self, tokens, seq_len):
        self.tokens = tokens
        self.seq_len = seq_len
        self.n = len(tokens) // (seq_len + 1)

    def __len__(self):
        return self.n

    def __getitem__(self, i):
        start = i * (self.seq_len + 1)
        w = self.tokens[start:start + self.seq_len + 1]
        # np.array(...) => copia (el memmap es read-only); son 2 KB por item
        return np.array(w[:-1], dtype=np.int32), np.array(w[1:], dtype=np.int32)

train_ds = TokenWindows(train_mm, SEQ_LEN)
valid_ds = TokenWindows(valid_mm, SEQ_LEN)
print('train: %s secuencias x %d tokens (%s tokens) | valid: %s'
      % (format(len(train_ds), ','), SEQ_LEN, format(len(train_ds) * SEQ_LEN, ','),
         format(len(valid_ds), ',')))

# generator con seed fijo => el orden del shuffle es reproducible (necesario para RESUME)
rng = torch.Generator().manual_seed(SEED)
train_dl = torch.utils.data.DataLoader(
    train_ds, batch_size=BATCH_SIZE, shuffle=True, drop_last=True,
    generator=rng, num_workers=0, pin_memory=(DEVICE == 'cuda'))
valid_dl = torch.utils.data.DataLoader(
    valid_ds, batch_size=EVAL_BATCH, shuffle=False, num_workers=0,
    pin_memory=(DEVICE == 'cuda'))
multi = 'DataParallel %d/GPU' % (BATCH_SIZE // NGPU) if NGPU > 1 else 'una GPU/CPU'
print('batches/epoch: %s | batch=%d (%s)' % (format(len(train_dl), ','), BATCH_SIZE, multi))


## 7️⃣ Construcción del modelo (~20.3M, contexto 512, cobertura binaria completa)

Se verifica el número real de parámetros y se **aborta si no es el modelo anunciado**
(antes, con `FAST_MODE=True` el notebook construía silenciosamente un modelo de 7.9M).


In [ ]:
config = EngramaConfig(vocab_size=VOCAB_SIZE, context_length=SEQ_LEN, **MODEL_KW)
torch.manual_seed(SEED)
model = EngramaModel(config).to(DEVICE)

n_params = model.num_parameters()
rf = config.receptive_field()
print('Parametros: %s  (~%.1fM)' % (format(n_params, ','), n_params / 1e6))
print('Campo receptivo: %s tokens | cubre contexto: %s'
      % (rf['max_reach'], rf['covers_context']))
print('Horizontes de cache jerarquico:', config.cache_horizons())

if not FAST_MODE and n_params < 19_000_000:
    raise RuntimeError(
        'MODELO INESPERADO: %s parametros (~%.1fM). Este notebook promete ~20.3M '
        'con FAST_MODE=False. Revisa MODEL_KW y FAST_MODE.'
        % (format(n_params, ','), n_params / 1e6))

class ShardLossModel(nn.Module):
    """Wrapper de DataParallel: cada GPU calcula la loss de SU trozo de batch.

    DataParallel solo reune escalares de loss, nunca logits completos (que con
    vocab 50,257 ocuparian ~3.3 GB extra en la GPU principal). La loss final es
    el promedio de los promedios por trozo = CE media global (trozo iguales).
    """
    def __init__(self, inner):
        super().__init__()
        self.inner = inner

    def forward(self, x, y):
        logits = self.inner(x)
        return chunked_cross_entropy(logits, y)

if NGPU > 1:
    train_model = nn.DataParallel(ShardLossModel(model), device_ids=list(range(NGPU)))
    print('Multi-GPU: DataParallel sobre %d GPUs (batch %d = %d por GPU)'
          % (NGPU, BATCH_SIZE, BATCH_SIZE // NGPU))
else:
    train_model = ShardLossModel(model)
    print('Entrenamiento en un solo dispositivo:', DEVICE)


## 8️⃣ Entrenamiento

AdamW + clipping + warmup lineal + decaimiento coseno, con evaluación periódica, muestreo de
generación y **guardado del mejor checkpoint por loss de validación** (+ estado para reanudar).

La evaluación y la pérdida usan `chunked_cross_entropy` (misma CE, sin materializar una segunda
copia de los logits) y el evocador `logsumexp` procesa el vocabulario por trozos con checkpointing.


In [ ]:
optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
os.makedirs(SAVE_DIR, exist_ok=True)

def lr_at(step):
    if step < WARMUP_STEPS:
        return LR * (step + 1) / WARMUP_STEPS
    p = (step - WARMUP_STEPS) / max(1, TOTAL_STEPS - WARMUP_STEPS)
    return LR * 0.5 * (1.0 + math.cos(math.pi * min(1.0, p)))

@torch.no_grad()
def evaluate(max_batches=EVAL_BATCHES):
    was_training = model.training
    model.eval()
    total, nb = 0.0, 0
    try:
        for i, (xb, yb) in enumerate(valid_dl):
            if i >= max_batches:
                break
            logits = model(xb.to(DEVICE))
            total += chunked_cross_entropy(logits, yb.to(DEVICE)).item() * xb.shape[0]
            nb += xb.shape[0]
    finally:
        if was_training:
            model.train()
    return total / max(1, nb)

def save_checkpoint(step, best):
    save_model(model, SAVE_DIR)  # config.json + model.pt (ultimo estado)
    torch.save({'optimizer': optimizer.state_dict(), 'global_step': step,
                'best_val': best}, os.path.join(SAVE_DIR, 'trainer_state.pt'))
    if _hf is not None:
        _hf.save_pretrained(os.path.join(SAVE_DIR, 'tokenizer'))

generator = Generator(model, tokenizer)
history = []
best_val = float('inf')
start_step = 0

if RESUME and os.path.exists(os.path.join(SAVE_DIR, 'model.pt')) \
        and os.path.exists(os.path.join(SAVE_DIR, 'trainer_state.pt')):
    try:
        model.load_state_dict(torch.load(os.path.join(SAVE_DIR, 'model.pt'),
                                         map_location=DEVICE))
        ck = torch.load(os.path.join(SAVE_DIR, 'trainer_state.pt'), map_location=DEVICE)
        optimizer.load_state_dict(ck['optimizer'])
        start_step = int(ck['global_step'])
        best_val = float(ck['best_val'])
        print('[resume] checkpoint encontrado: paso %d, best_val %.4f' % (start_step, best_val))
        skip = start_step % max(1, len(train_dl))
        if skip:
            print('[resume] omitiendo %d batches ya vistos (mismo seed => mismo orden) ...' % skip)
            for _ in range(skip):
                next(iter(train_dl))
    except Exception as exc:
        print('[resume] no reanudable (%s); se entrena desde cero.' % type(exc).__name__)

TOTAL_STEPS = start_step + len(train_dl) * EPOCHS
new_steps = len(train_dl) * EPOCHS
print('%d pasos nuevos (total %d contando resume)' % (new_steps, TOTAL_STEPS))
if not FAST_MODE:
    print('Estimacion: ~1-2 s/paso en T4 x2 (batch 32) -> %.1f h aprox.'
          % (new_steps * 1.5 / 3600))

global_step = start_step
t0 = time.time()
model.train()
for epoch in range(EPOCHS):
    for xb, yb in train_dl:
        lr = lr_at(global_step)
        for gparam in optimizer.param_groups:
            gparam['lr'] = lr
        loss = train_model(xb.to(DEVICE), yb.to(DEVICE))
        if loss.dim() > 0:          # DataParallel devuelve un escalar por GPU
            loss = loss.mean()
        optimizer.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP)
        optimizer.step()
        history.append((global_step, loss.item()))

        if global_step % LOG_EVERY == 0:
            print('paso %6d/%d | loss %.4f | lr %.2e | %ds'
                  % (global_step, TOTAL_STEPS, loss.item(), lr, int(time.time() - t0)))
        if (global_step + 1) % EVAL_EVERY == 0 or global_step + 1 == TOTAL_STEPS:
            val = evaluate()
            print('  [eval] paso %d: val_loss %.4f | val_ppl %.2f'
                  % (global_step + 1, val, math.exp(min(20, val))))
            if val < best_val:
                best_val = val
                torch.save(model.state_dict(), os.path.join(SAVE_DIR, 'best_model.pt'))
                print('  [ckpt] mejor checkpoint guardado (val %.4f)' % best_val)
        if (global_step + 1) % SAMPLE_EVERY == 0:
            model.eval()
            print('  [muestra]', repr(generator.generate(
                'Once upon a time', max_new_tokens=64, temperature=0.8,
                top_k=40, stop_at_eos=True)))
            model.train()
        global_step += 1

print('Entrenamiento terminado en %ds | mejor val_loss %.4f'
      % (int(time.time() - t0), best_val))
if DEVICE == 'cuda':
    for gi in range(NGPU):
        print('  GPU %d memoria pico: %.2f GiB'
              % (gi, torch.cuda.max_memory_allocated(gi) / 2**30))


### Curva de pérdida


In [ ]:
try:
    import matplotlib.pyplot as plt
    steps, losses = zip(*history)
    plt.figure(figsize=(8, 4))
    plt.plot(steps, losses, alpha=0.3, label='train loss')
    w = max(5, len(losses) // 50)
    plt.plot(steps[w - 1:], [sum(losses[i - w:i]) / w for i in range(w, len(losses) + 1)],
             label='media movil (%d)' % w)
    plt.xlabel('paso'); plt.ylabel('cross-entropy'); plt.legend(); plt.grid(alpha=0.3)
    plt.title('ENGRAMA V3 ~20M en TinyStories (GPT-2, seq 512)')
    plt.show()
except ImportError:
    print('matplotlib no disponible; historial:', history[:3], '...', history[-3:])


## 9️⃣ Evaluación final y guardado


In [ ]:
os.makedirs(SAVE_DIR, exist_ok=True)
final_val = evaluate(max_batches=len(valid_dl))
print('Validacion completa: loss %.4f | perplejidad %.2f'
      % (final_val, math.exp(min(20, final_val))))

if final_val < best_val:
    best_val = final_val
    torch.save(model.state_dict(), os.path.join(SAVE_DIR, 'best_model.pt'))
save_checkpoint(global_step, best_val)   # ultimo estado + optimizer (para RESUME)
print('Checkpoint en', SAVE_DIR, ':', sorted(os.listdir(SAVE_DIR)))

with open(os.path.join(SAVE_DIR, 'training_log.json'), 'w') as f:
    json.dump({'params': n_params, 'seq_len': SEQ_LEN, 'vocab': VOCAB_SIZE,
               'steps': TOTAL_STEPS, 'train_sequences': len(train_ds),
               'best_val_loss': best_val, 'final_val_loss': final_val,
               'fast_mode': FAST_MODE, 'seed': SEED}, f, indent=2)
print('training_log.json escrito')


## 🔟 Inferencia con el modelo entrenado

Recarga del checkpoint desde disco (prueba de persistencia completa) y generación con
temperatura / top-k / top-p y parada en `<|endoftext|>`.


In [ ]:
loaded_model, _ = load_model(SAVE_DIR, device=DEVICE)
weights = 'best_model.pt' if os.path.exists(os.path.join(SAVE_DIR, 'best_model.pt')) else 'model.pt'
if weights != 'model.pt':
    loaded_model.load_state_dict(torch.load(os.path.join(SAVE_DIR, weights),
                                            map_location=DEVICE))
loaded_model.eval()

tok = tokenizer
if _hf is not None and os.path.isdir(os.path.join(SAVE_DIR, 'tokenizer')):
    try:
        from transformers import GPT2TokenizerFast
        tok = GPT2Adapter(GPT2TokenizerFast.from_pretrained(
            os.path.join(SAVE_DIR, 'tokenizer')))
        print('Tokenizer recargado desde el checkpoint.')
    except Exception as exc:
        print('Tokenizer del checkpoint no disponible (%s); uso el de memoria.' % type(exc).__name__)

gen = Generator(loaded_model, tok)
print('Checkpoint recargado:', format(loaded_model.num_parameters(), ','), 'parametros (%s)' % weights)
print()
for prompt in ['Once upon a time', 'One day, a little girl named Anna',
               'Tom found a big red ball']:
    out = gen.generate(prompt, max_new_tokens=120, temperature=0.8,
                       top_k=40, top_p=0.95, stop_at_eos=True)
    print('==>', prompt)
    print(out.replace('<|endoftext|>', '').strip()[:600], '\n')


### Invarianza causal del checkpoint (verificación rápida)


In [ ]:
xb, _ = next(iter(valid_dl))
x = xb[:2, :64].to(DEVICE)
with torch.no_grad():
    full = loaded_model(x)
    for mode in ('full', 'hierarchical'):
        cache = loaded_model.get_cache(N_max=64, mode=mode)
        steps = [loaded_model.step_forward(x[:, t:t + 1], cache, t)[0]
                 for t in range(64)]
        inc = torch.stack(steps, dim=1)
        print('cache %-13s: max |diff| = %.2e  (< 1e-4 OK)'
              % (mode, (full - inc).abs().max().item()))


## ✅ Notas honestas

- **FAST_MODE=False por defecto**: TinyStories completo (~370k secuencias de 512), ~11k pasos
  con batch 32 en T4 x2 (unas 3-6 h). El notebook **aborta** si construye un modelo distinto al
  anunciado o si el dataset no puede descargarse completo: nada de sustituciones silenciosas.
- **Memoria**: la causa del OOM anterior no era el modelo (20M es pequeño) sino los logits del
  evocador `logsumexp`: `(16, 512, 4 candidatos, 50257)` en fp32 son ~6.6 GB por GPU, y con los
  temporales del softmax se superaban los 16 GB de la T4. Ahora el vocabulario se procesa por
  trozos con checkpointing y la CE por trozos: pico medido ~3-4 GiB/GPU.
- Multi-GPU: se usa `DataParallel` por ser robusto dentro de un notebook (DDP necesitaría lanzar
  procesos, frágil en Kaggle). La pérdida se calcula por GPU, así que no hay gather de logits.
  Evaluación, generación y guardado usan siempre el modelo base (checkpoints portables a CPU).
- Calidad esperada en FULL: TinyStories es un benchmark amable para modelos de ~20M (perplejidad
  de validación típicamente < 10 con datos suficientes); el objetivo es demostrar el pipeline
  completo de ENGRAMA V3 end-to-end, no batir el estado del arte.
- Reproducibilidad: `training_log.json` registra semilla, pasos, pérdidas y configuración;
  `RESUME=True` reanuda desde el último paso guardado con el mismo orden de datos.
- Licencia AGPL-3.0 · Autor: BUEORM · [github.com/bueormnew/engrama](https://github.com/bueormnew/engrama)
